# 3. Регрессия CC50

In [ ]:
# Подключение библиотек

from pathlib import Path
import json
import warnings

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

random_state = 42
target_columns = ["IC50, mM", "CC50, mM", "SI"]

data_path = Path("/Users/dmitrijgoncarov/mephi_masters/sem_2/classic_ML/coursework/drug_activity_prediction_variant/notebooks/coursework_dataset.csv")
results_dir = Path("/Users/dmitrijgoncarov/mephi_masters/sem_2/classic_ML/coursework/drug_activity_prediction_variant/notebooks/results")
figures_dir = results_dir / "figures"
tables_dir = results_dir / "tables"
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Загрузка данных

dataset = pd.read_csv(data_path)
dataset = dataset.drop(columns=[column for column in dataset.columns if str(column).startswith("Unnamed")], errors="ignore")
dataset = dataset.apply(pd.to_numeric, errors="coerce")
dataset = dataset.drop_duplicates()
dataset = dataset.dropna(subset=target_columns).reset_index(drop=True)

print("Размер датасета после базовой очистки:", dataset.shape)
print(dataset.head())


In [ ]:
# Подготовка признаков и целевой переменной

target_column = 'CC50, mM'
task_id = 'regression_cc50'

feature_columns = [column for column in dataset.columns if column not in target_columns]
x = dataset[feature_columns]
y = dataset[target_column].astype(float)

print("Целевая переменная:", target_column)
print("Количество признаков:", x.shape[1])
print(y.describe())


## Методический вывод

Из матрицы признаков исключены IC50, CC50 и SI, чтобы модель использовала только молекулярные дескрипторы. Это особенно важно для CC50, так как показатель связан с общей структурой молекулы и может быть косвенно связан с другими целевыми переменными.

Для обучения используется единый конвейер: заполнение пропусков медианой, удаление константных признаков, масштабирование для чувствительных моделей и подбор гиперпараметров по кросс-валидации. Такой подход делает сравнение моделей воспроизводимым.

In [ ]:
# Разбиение данных и описание конвейера обучения

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=random_state,
)


def make_pipeline(model, scale=False):
    # Конвейер выполняет одинаковую предобработку на кросс-валидации и тесте.
    # Медианная импутация обрабатывает пропуски без удаления наблюдений.
    # VarianceThreshold удаляет константные признаки.
    # RobustScaler применяется для моделей, чувствительных к масштабу признаков.
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold()),
    ]
    if scale:
        steps.append(("scaler", RobustScaler()))
    steps.append(("model", model))
    return Pipeline(steps)


base_models = {
    "Ridge": (
        make_pipeline(Ridge(random_state=random_state), scale=True),
        {"regressor__model__alpha": [0.1, 1.0, 10.0, 100.0]},
    ),
    "KNN": (
        make_pipeline(KNeighborsRegressor(), scale=True),
        {"regressor__model__n_neighbors": [5, 15]},
    ),
    "RandomForest": (
        make_pipeline(RandomForestRegressor(random_state=random_state, n_jobs=-1), scale=False),
        {"regressor__model__n_estimators": [120], "regressor__model__max_depth": [None, 12]},
    ),
    "ExtraTrees": (
        make_pipeline(ExtraTreesRegressor(random_state=random_state, n_jobs=-1), scale=False),
        {"regressor__model__n_estimators": [140], "regressor__model__max_depth": [None, 12]},
    ),
    "GradientBoosting": (
        make_pipeline(GradientBoostingRegressor(random_state=random_state), scale=False),
        {"regressor__model__n_estimators": [100, 160], "regressor__model__learning_rate": [0.05, 0.1]},
    ),
}

models = {
    model_name: (
        TransformedTargetRegressor(regressor=model_pipeline, func=np.log1p, inverse_func=np.expm1),
        param_grid,
    )
    for model_name, (model_pipeline, param_grid) in base_models.items()
}


In [ ]:
# Обучение моделей и подбор гиперпараметров

cross_validation = KFold(n_splits=3, shuffle=True, random_state=random_state)
metric_rows = []
best_estimator = None
best_rmse = np.inf

for model_name, (model, param_grid) in models.items():
    print("Обучение модели:", model_name)

    search = GridSearchCV(
        model,
        param_grid,
        scoring="neg_root_mean_squared_error",
        cv=cross_validation,
        n_jobs=-1,
    )
    search.fit(x_train, y_train)

    prediction = search.best_estimator_.predict(x_test)
    rmse = root_mean_squared_error(y_test, prediction)
    mae = mean_absolute_error(y_test, prediction)
    r2 = r2_score(y_test, prediction)

    metric_rows.append(
        {
            "model": model_name,
            "cv_rmse": -search.best_score_,
            "test_rmse": rmse,
            "test_mae": mae,
            "test_r2": r2,
            "best_params": json.dumps(search.best_params_, ensure_ascii=False),
        }
    )

    if rmse < best_rmse:
        best_rmse = rmse
        best_estimator = search.best_estimator_

metrics = pd.DataFrame(metric_rows).sort_values("test_rmse").reset_index(drop=True)
metrics_path = tables_dir / f"{task_id}_metrics.csv"
metrics.to_csv(metrics_path, index=False)

print(metrics)
print("Метрики сохранены:", metrics_path)


## Вывод по результатам моделирования

Для CC50 лучший результат показала модель RandomForest: RMSE = 461.797, MAE = 276.525, R2 = 0.470. Это лучший показатель R2 среди трех регрессионных задач, поэтому CC50 прогнозируется устойчивее, чем IC50 и SI.

Вероятная причина состоит в том, что CC50 сильнее связан с размерными и структурными дескрипторами молекулы: массой, количеством тяжелых атомов, площадью поверхности и близкими признаками. Такие зависимости хорошо обрабатываются ансамблями деревьев.

In [ ]:
# Сравнение моделей по тестовой RMSE

plot_data = metrics.sort_values("test_rmse", ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(plot_data["model"], plot_data["test_rmse"])
plt.gca().invert_yaxis()
plt.xlabel("Test RMSE")
plt.title(f"Сравнение моделей: {task_id}")
plt.tight_layout()
comparison_path = figures_dir / f"{task_id}_model_comparison.png"
plt.savefig(comparison_path, dpi=160)
plt.show()

print("График сохранен:", comparison_path)


In [ ]:
# Диагностика лучшей модели

best_prediction = best_estimator.predict(x_test)

plt.figure(figsize=(6, 5))
plt.scatter(y_test, best_prediction, alpha=0.65)
lower_value = min(float(y_test.min()), float(np.min(best_prediction)))
upper_value = max(float(y_test.max()), float(np.max(best_prediction)))
plt.plot([lower_value, upper_value], [lower_value, upper_value], color="crimson", linewidth=2)
plt.xlabel("Фактическое значение")
plt.ylabel("Прогноз")
plt.title(f"{task_id}: прогноз против факта")
plt.tight_layout()
scatter_path = figures_dir / f"{task_id}_predicted_vs_actual.png"
plt.savefig(scatter_path, dpi=160)
plt.show()

print("График сохранен:", scatter_path)
print("Лучшая модель:", metrics.loc[0, "model"])


## Итоговый аналитический вывод

Регрессия CC50 является наиболее успешной из трех регрессионных задач. Модель не идеально прогнозирует отдельные экстремальные значения, но в среднем лучше улавливает зависимость целевой переменной от дескрипторов. Для дальнейшего улучшения результата можно исследовать важность признаков RandomForest и удалить избыточные сильно коррелирующие дескрипторы.